# Lab 7.1 — First Calls with the OpenAI API  *(SOLUTION / instructor copy)*

**Chapter 7 — Building with LLMs: APIs, RAG & Agents**

This is the rewrite of the former Lab 6.3. It uses the current OpenAI Python SDK
(v1.x+), reads the API key from the environment (already set up for you on the
VM — there is no key to paste), and pins the model through a single variable so
the lab keeps working as models change.

**Objectives**
1. Make a working chat completion call from Python.
2. See how the *system* prompt and *temperature* change the output.
3. Get **structured (JSON) output** you can use in a program.
4. Read the token usage and estimate cost at agency scale.

## Setup — no key to paste

On the classroom VM, `OPENAI_API_KEY` and `OPENAI_MODEL` are already set — the
helpers create the client and pin the model in one place. If you ever move this
pattern to Azure OpenAI or Amazon Bedrock in production, only the setup
changes.

In [1]:
# Instructor copies live in solutions/, one level below labs/ — find labs/
# (where lab_common.py, lab_helpers.py and data/ are) and run from there.
import os, sys
from pathlib import Path

for _cand in (Path.cwd(), *Path.cwd().parents):
    if (_cand / "lab_common.py").is_file():
        os.chdir(_cand)
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        break

from lab_helpers import *

## Exercise 1 — Your first chat call

The Chat Completions API takes a list of *messages*. Each message has a `role`
(`system`, `user`, or `assistant`) and `content` — the helper sends the
student's question as the single user message.

In [2]:
QUESTION = "In two sentences, what is the Freedom of Information Act?"

first_foia_call(QUESTION)

(offline) canned reply from the instructor transcript:

The Freedom of Information Act (FOIA) is a federal law that gives any person the right to request records from U.S. executive-branch agencies. Agencies must generally respond within 20 business days, releasing records unless one of nine exemptions applies — for example personal privacy, law-enforcement sensitivity, or national security.


'The Freedom of Information Act (FOIA) is a federal law that gives any person the right to request records from U.S. executive-branch agencies. Agencies must generally respond within 20 business days, releasing records unless one of nine exemptions applies — for example personal privacy, law-enforcement sensitivity, or national security.'

## Exercise 2 — The system prompt changes the voice

Same question, different `system` message. Notice how the persona shifts the
tone without changing the facts.

In [3]:
FOIA_QUESTION = "How quickly must an agency respond to a FOIA request?"

FORMAL_PERSONA = "You are a policy analyst. Answer formally, citing the 20-business-day response rule."

PLAIN_PERSONA = "You are explaining to a brand-new employee. Answer in plain, friendly language."

compare_foia_voices(FOIA_QUESTION, FORMAL_PERSONA, PLAIN_PERSONA)

(offline) canned replies from the instructor transcript:

FORMAL REGISTER:
 Under the Freedom of Information Act (5 U.S.C. § 552), an agency must determine whether to comply with a request within twenty (20) business days of receipt. In unusual circumstances the agency may extend this period by up to ten additional business days, provided it notifies the requester in writing.

PLAIN-LANGUAGE REGISTER:
 Think of it as a 20-working-day clock: once your FOIA request arrives, the agency has about four weeks of business days to get back to you. If things get complicated it can take a bit longer, but it has to tell you first.


('Under the Freedom of Information Act (5 U.S.C. § 552), an agency must determine whether to comply with a request within twenty (20) business days of receipt. In unusual circumstances the agency may extend this period by up to ten additional business days, provided it notifies the requester in writing.',
 'Think of it as a 20-working-day clock: once your FOIA request arrives, the agency has about four weeks of business days to get back to you. If things get complicated it can take a bit longer, but it has to tell you first.')

## Exercise 3 — Temperature

`temperature` controls randomness. Low (0–0.3) = focused and repeatable; high
(0.8–1.2) = varied and creative. Run this twice and compare.

Note the model: reasoning models such as the gpt-5 family accept only the
default temperature and reject anything else, so this one exercise pins a
temperature-capable model. That is itself worth knowing — "which knobs exist"
is a property of the model, not of the API.

In [4]:
TITLE_PROMPT = "Suggest a title for a one-page guide on using AI responsibly in government."

try_temperature_titles(TITLE_PROMPT)

(offline) canned titles from the instructor transcript:

temperature=0.0: Responsible AI in Government: A One-Page Guide
temperature=1.0: AI With Guardrails: A Field Guide for Public Servants

Run the cell again. At 0.0 the title should barely move; at 1.0 it should.


{'0.0': 'Responsible AI in Government: A One-Page Guide',
 '1.0': 'AI With Guardrails: A Field Guide for Public Servants'}

## Exercise 4 — Structured output (JSON) from a real document

Programs need structured data, not prose. We ask the model to extract fields
from the interim-guidance memo as JSON, and enforce it with
`response_format={"type": "json_object"}` (the helper applies it).

**Worked instruction text** below — the elements that make it reliable:
naming the keys, giving each a type, and saying "ONLY".

In [5]:
EXTRACTION_INSTRUCTIONS = (
    "Extract fields from the memo. Reply ONLY with JSON having keys: "
    "subject (string), effective_date (string, YYYY-MM-DD), "
    "key_rules (array of short strings)."
)

extraction = extract_memo_json_71(EXTRACTION_INSTRUCTIONS)

(offline) canned extraction from the instructor transcript:

{
  "subject": "Interim Guidance on Generative AI for Constituent Services",
  "effective_date": "2026-03-14",
  "key_rules": [
    "Every AI-assisted work product must be reviewed by a responsible employee before release; the employee, not the tool, is accountable.",
    "Only public information may be entered into public AI tools; PII must never be entered into an unapproved tool, and exposure is reported within one business day.",
    "Tasks involving internal information must use the enterprise assistant on the approved-tools list maintained by the CIO.",
    "AI-assisted correspondence documenting agency business is a federal record and must be retained on the applicable schedule.",
    "Correspondence drafted with AI assistance must disclose that in a closing line, and the tool and reviewer are logged in the case system."
  ]
}

✓ all three keys present; key_rules has 5 entries


## Exercise 5 — Token usage and cost

Every response reports token usage. This is how you estimate what a workload
costs before you scale it to an agency. Offline, the numbers below are the
ones captured from a live instructor run — the arithmetic is what students
must reproduce.

**Expected:** prompt 631 + completion 174 = 805 tokens → ≈ $0.00020 per call,
≈ $0.20 per 1,000 calls at the illustrative rates.

In [6]:
show_token_cost()

(illustrative token counts from the instructor transcript — a live run shows your own)

prompt 631 + completion 174 = 805 tokens
≈ $0.00020 per call  →  $0.20 per 1,000 calls


## Stretch A — Tighten the prompt until it validates every time

**Expected:** 3/3 VALID with the worked instructions above. When a student's
instructions validate only intermittently, the fix is almost always naming
the keys explicitly and forbidding extra text.

In [7]:
validate_extraction_runs(EXTRACTION_INSTRUCTIONS)

run 1: VALID
run 2: VALID
run 3: VALID

3/3 valid — keep tightening until it is 3/3.


## Debrief
1. Which changed the output more — the system prompt or the temperature? Why?
2. Why is JSON output (Exercise 4) more useful in a real system than prose?
3. Your agency wants to summarize 50,000 documents a month. Using the cost from
   Exercise 5, roughly what would that cost — and what would change if you ran it
   on Azure OpenAI (FedRAMP) instead?

**What changed from the old lab:** the retired `openai.Completion.create(...)`
call and the `text-davinci-003` model are gone, and there is no `api_key =
"your_api_key_here"` line — the key comes from the environment.